# Module 06: Pandas, continued, and Intro to Seaborn

Follow along here during class. Feel free to add your own cells and notes.

The git section at the start of today's class (`git pull`, `git rebase`) happens in your terminal, not in this notebook.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/gapminder.tsv', sep='\t')

## Pandas, continued

### Selecting Data: `.iloc`

`.iloc` uses numeric positions for the rows and columns.

In [ ]:
df.iloc[:100]

In [ ]:
df.iloc[:100, :2]

### Selecting Data: `.loc`

`.loc` uses the row names and column names. Unlike `.iloc`, `.loc` includes the end of the slice.

In [ ]:
df.loc[:100]

In [ ]:
df.loc[:100, ['continent', 'country']]

`.loc` also accepts a Boolean expression for the rows, with the columns you want after the comma.

In [ ]:
df.loc[df['year'] == 2007, ['country', 'lifeExp']]

### Practice (5 min)

Let's work on these questions:

1. Select the first 5 rows of the `country` and `continent` columns, once with `.iloc` and once with `.loc`.
2. Select rows 10 through 20 with `df.iloc[10:20]` and with `df.loc[10:20]`. Why is the number of rows different?
3. Use `.loc` to select the `country` and `lifeExp` columns for all rows where `continent` is `'Oceania'`.

In [ ]:
# 1.


In [ ]:
# 2.


In [ ]:
# 3.


### Group-Apply-Combine

1. **Group**: split the rows into groups based on a column
2. **Apply**: compute something on each group, e.g. the mean
3. **Combine**: put the results back together in one table

### Group By

Let's go back to the zoo from Module 5.

In [ ]:
my_zoo = {
    'animal': ['panda', 'panda', 'lion', 'tiger', 'lion'],
    'sex': ['f', 'm', 'f', 'f', 'm'],
    'age': [1, 5, 2.5, 3, 3]
}

zoo = pd.DataFrame(my_zoo)

If you want to find a breakdown of a given level within your dataframe, you can use the groupby method to specify what column you're interested in grouping by.

This lets you compute summary statistics for the given grouping:

In [ ]:
zoo.groupby('animal')['age'].mean()

You can also group by multiple columns:

In [ ]:
zoo.groupby(['sex', 'age']).count()

### Groupby

We use `.groupby` method when we want to do calculations based on a particular group. This answers the question from Module 5: what is the average life expectancy for a given country?

In [ ]:
df.groupby('country')['lifeExp'].mean()

By default, it will arrange the countries in alphabetic order. But if we wanted to group based on the continent as well:

In [ ]:
df.groupby(['continent', 'country'])['lifeExp'].mean()

### Aggregate

If you would like to calculate different summary statistics for your dataframe, you can do so by using the `.agg()` method.

In [ ]:
df.groupby(['continent']).agg({
    'lifeExp': ['mean', 'median', 'max']
})

Grouping by more than one column gives a Series with a multi-level index. Use `.reset_index()` to turn the groups back into regular columns.

In [ ]:
df.groupby(['continent', 'year'])['lifeExp'].mean().reset_index()

### Practice (5 min)

Let's work on these questions:

1. Compute the mean `gdpPercap` for each continent.
2. Compute the mean and median `lifeExp` for each `year`. How has life expectancy changed over time?
3. Group by `continent` and `year`, then use `.reset_index()`. Which continent had the highest mean `lifeExp` in `2007`?

In [ ]:
# 1.


In [ ]:
# 2.


In [ ]:
# 3.


### Grouping Needs a Column

To group by something, the information has to be in a column. Here are read counts from an RNA-seq experiment, with the sample names as columns. `FB` and `HSC` are cell types, and `_1` and `_2` are replicates.

In [ ]:
counts = pd.DataFrame({
    'gene': ['Xkr4', 'Rp1'],
    'FB_1': [2, 0], 'FB_2': [0, 6],
    'HSC_1': [0, 2], 'HSC_2': [0, 0]
})
counts

### Wide to Long: `.melt()`

`.melt()` turns the sample columns into rows, so that the sample name is now a column.

In [ ]:
long = counts.melt(id_vars='gene', var_name='sample', value_name='count')
long

`.str.split()` splits the text in a column. `expand=True` puts each piece in its own column.

In [ ]:
long[['group', 'replicate']] = long['sample'].str.split('_', expand=True)
long

In [ ]:
long.groupby('group')['count'].mean()

### Practice (5 min)

Let's work on these questions:

1. Run `.melt()` on `counts`. What is the `shape` before and after? Why?
2. Use `.str.split()` to add `group` and `replicate` columns.
3. Compute the mean `count` for each `group`.

In [ ]:
# 1.


In [ ]:
# 2.


In [ ]:
# 3.


### Renaming Columns

`.rename()` takes a dictionary of `{old name: new name}`. To prevent overwriting your dataframe, Pandas creates a copy of your dataframe, so we save it back to `df`. From here on, we'll use `life_expectancy` and `gdp_per_capita`.

In [ ]:
df = df.rename(
    columns = {
        'lifeExp': 'life_expectancy',
        'gdpPercap': 'gdp_per_capita'
})
df.columns

If you want to change every column name, you can set `df.columns` to a list of names instead. The list has to be the same length as the columns, and in the same order.

In [ ]:
df.columns = ['country', 'continent', 'year', 'life_expectancy', 'pop', 'gdp_per_capita']

### Practice (5 min)

Let's work on these questions:

1. Save a copy called `df_renamed` with `pop` renamed to `population`, and check its `columns`.
2. Rename the `gene` column of `counts` to `gene_name`.
3. Why is `df.rename(columns={'pop': 'population'})` on its own not enough to change `df`?

In [ ]:
# 1.


In [ ]:
# 2.


In [ ]:
# 3.


### Cheatsheet

https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf

## Intro to Seaborn

Seaborn is a high-level visualization library that works on top of matplotlib. By convention, we import it as `sns`, which we did at the top of this notebook.

### A Basic Statistical Plot

`sns.histplot()` plots the distribution of your data. `kde=True` adds a smoothed estimate of the distribution on top.

In [ ]:
rng = np.random.default_rng(seed=42)
measurements = rng.normal(loc=100, scale=15, size=1_000)

sns.histplot(x=measurements, kde=True)

### Seaborn and Matplotlib Together

We can pass a matplotlib `Axes` to target the location of the plots. The Axes methods we learned, e.g. `.set_title()`, still work on seaborn plots.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3), layout='constrained')
sns.histplot(x=measurements, ax=axes[0])
sns.kdeplot(x=measurements, ax=axes[1])
axes[0].set_title('Histogram')
axes[1].set_title('KDE')

### Comparing Groups

Pass a 2-d array to `data`, and seaborn treats each column as a group.

In [ ]:
groups = rng.normal(loc=[0, 1, 2], scale=1, size=(200, 3))

fig, axes = plt.subplots(1, 2, figsize=(8, 3), layout='constrained')
sns.boxplot(data=groups, ax=axes[0])
sns.violinplot(data=groups, ax=axes[1])

### Seaborn with DataFrames

Pass the DataFrame to `data` and name the columns to use for `x`, `y`, and more. Seaborn does the grouping for us, one box for each continent.

In [ ]:
df_2007 = df.query("year == 2007")

sns.boxplot(data=df_2007, x='continent', y='life_expectancy')

Seaborn lets us convenient make plots and we can pass matplotlib `Axis` to target the location of the plots.

In [ ]:
fig, ax = plt.subplots(1, 2)
sns.lineplot(x='year', y='life_expectancy', data=df, ax=ax[0], linestyle=':')
sns.lineplot(x='year', y='life_expectancy', data=df, ax=ax[1], linestyle='-.')

`hue` colors the points by a column. Wealth spans orders of magnitude, so a log axis spreads the countries out.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), layout='constrained')
sns.scatterplot(data=df_2007, x='gdp_per_capita', y='life_expectancy', hue='continent', ax=ax)
ax.set_xscale('log')

`sns.lineplot()` draws a line for each value of `hue`. When there are many rows per `x`, seaborn plots the mean and a shaded interval around it.

In [ ]:
sns.lineplot(data=df, x='year', y='life_expectancy', hue='continent')

### Practice (5 min)

Let's work on these questions:

1. Simulate rolling two dice 10,000 times, like in Module 3, and plot the distribution of the sum with `sns.histplot(..., discrete=True)`.
2. Simulate three groups with different standard deviations and compare them with `sns.boxplot()` and `sns.violinplot()` side by side.
3. Label your axes, add a figure title, and save your figure.

In [ ]:
# 1.


In [ ]:
# 2.


In [ ]:
# 3.


### Practice (10 min)

Let's work on these questions:

1. Use `sns.histplot()` on `df_2007` to plot the distribution of `life_expectancy`. Add `hue='continent'`. What do you notice?
2. Make a scatterplot of `gdp_per_capita` against `life_expectancy` for `df_2007`, with the size of the points set by `pop` (`size='pop'`).
3. Use `sns.lineplot()` to show `pop` over `year`, with a line for each `continent`. Which continent grew the fastest?

In [ ]:
# 1.


In [ ]:
# 2.


In [ ]:
# 3.


### Exercise

Let's do some more exploration of the Gapminder dataset.

- Create a breakdown by year and country of the average life expectancy and the average GDP per capita
- Create another breakdown by year and continent
- Create a breakdown by year for average life expectancy and GDP per capita
- Create a figure with two subplots with the two calculated averages

In [ ]:
# 1.


In [ ]:
# 2.


In [ ]:
# 3.


In [ ]:
# 4.
